# 3. Validate tumor and macrophage annotations with InferCNV

**Objective:** Infer CNVs with infercnvpy to determine whether clusters 5 and 7 correspond to macrophages or tumor cells.
- Tumor cells are expected to show clear CNVs, such as the characteristic GBM pattern of chr7 gain and chr10 loss.  
- Immune cells are expected to remain largely diploid.

In [ ]:
import scanpy as sc
import infercnvpy as cnv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sc.set_figure_params(dpi=100, frameon=False)
%config InlineBackend.figure_format = 'retina'

FIG_DIR = Path('./figures/3_CNV_validation')
FIG_DIR.mkdir(parents=True, exist_ok=True)

### 1. Load the data

In [ ]:
adata = sc.read_h5ad('./write/0_integrate_two_datasets/adata2.h5ad')
print(f'{adata.shape[0]} cells × {adata.shape[1]} genes')
print(adata.obs['cell_type'].value_counts())

### 2. Annotate gene positions using the GENCODE v44 GTF

In [ ]:
gtf_path = '../snGBM/results/gencode.v44.basic.annotation.gtf.gz'
cnv.io.genomic_position_from_gtf(gtf_path, adata=adata, gtf_gene_id='gene_name')

# Check chromosome annotation coverage.
has_chr = adata.var['chromosome'].notna()
print(f'Genes with chromosome annotations: {has_chr.sum()} / {adata.n_vars} ({has_chr.mean():.1%})')
print(adata.var[has_chr][['chromosome', 'start', 'end']].head(10))

### 3. Run InferCNV

Use Endothelial, NK/T and Pericyte as the normal reference groups, selected as the most reliable nonmalignant populations.

In [ ]:
# Use vascular and lymphoid cells as normal references because they are least likely to be contaminated.
ref_types = ['Endothelial', 'NK/T', 'Pericyte']
n_ref = adata.obs['cell_type'].isin(ref_types).sum()
print(f'Reference cells: {n_ref} ({ref_types})')

cnv.tl.infercnv(
    adata,
    reference_key='cell_type',
    reference_cat=ref_types,
    window_size=250,
    step=10,
    dynamic_threshold=1.5,
    n_jobs=4,
)
print('✅ InferCNV complete')
print(f'CNV matrix shape: {adata.obsm["X_cnv"].shape}')

### 4. Calculate CNV scores

In [ ]:
# In CNV space, run PCA, construct the neighbor graph, cluster with Leiden and calculate the score.
cnv.tl.pca(adata)
cnv.pp.neighbors(adata)
cnv.tl.leiden(adata, resolution=0.5)

cnv.tl.cnv_score(adata)
print(f'CNV score range: {adata.obs["cnv_score"].min():.4f} - {adata.obs["cnv_score"].max():.4f}')
print('\nCNV score by cell_type:')
print(adata.obs.groupby('cell_type')['cnv_score'].agg(['mean', 'median', 'std']).sort_values('mean', ascending=False).round(4))

### 5. Visualize CNV scores

In [ ]:
# CNV score on UMAP
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sc.pl.umap(adata, color='cnv_score', ax=axes[0], frameon=False,
           title='CNV Score', cmap='RdBu_r', vcenter=0, show=False)
sc.pl.umap(adata, color='cell_type', ax=axes[1], frameon=False,
           title='Cell Type', legend_loc='on data', legend_fontsize=7, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'umap_cnv_score.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Box plots of CNV scores by cell_type
order = adata.obs.groupby('cell_type')['cnv_score'].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=adata.obs, x='cell_type', y='cnv_score', order=order, ax=ax, fliersize=0.5)
ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title('CNV Score by Cell Type')
plt.tight_layout()
plt.savefig(FIG_DIR / 'boxplot_cnv_score_by_celltype.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show CNV scores by Leiden cluster and label each cluster with its cell_type.
df_plot = adata.obs[['leiden', 'cell_type', 'cnv_score']].copy()
df_plot['label'] = df_plot['leiden'].astype(str) + ' (' + df_plot['cell_type'].astype(str) + ')'
order_cl = df_plot.groupby('label')['cnv_score'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(16, 5))
sns.boxplot(data=df_plot, x='label', y='cnv_score', order=order_cl, ax=ax, fliersize=0.3)
ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha='right', fontsize=8)
ax.set_title('CNV Score by Leiden Cluster (annotated cell type)')

# Highlight clusters 5 and 7.
for i, label in enumerate(order_cl):
    if label.startswith('5 ') or label.startswith('7 '):
        ax.get_xticklabels()[i].set_color('red')
        ax.get_xticklabels()[i].set_fontweight('bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'boxplot_cnv_score_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

### 6. Chromosome-level CNV heatmaps

In [ ]:
# Chromosome-level CNV heatmap grouped by cell_type
cnv.pl.chromosome_heatmap_summary(adata, groupby='cell_type', show=True, 
                                   save='_celltype.png')

In [ ]:
# Chromosome-level CNV heatmap grouped by Leiden cluster
cnv.pl.chromosome_heatmap_summary(adata, groupby='leiden', show=True,
                                   save='_leiden.png')

### 7. Compare clusters 5 and 7 with known tumor and immune clusters

In [ ]:
# Compare known tumor clusters (8/9), known immune clusters (0/6/3) and the clusters under evaluation (5/7).
focus_clusters = ['8', '9', '0', '6', '3', '5', '7', '2', '4']
adata_focus = adata[adata.obs['leiden'].isin(focus_clusters)].copy()

# Add comparison group labels.
group_map = {
    '8': 'Known_Tumor', '9': 'Known_Tumor',
    '0': 'Known_Immune', '6': 'Known_Immune', '3': 'Known_Immune',
    '2': 'Macrophage_assigned', '4': 'Macrophage_assigned',
    '5': 'Verify_cl5', '7': 'Verify_cl7',
}
adata_focus.obs['verify_group'] = adata_focus.obs['leiden'].map(group_map).astype('category')

fig, ax = plt.subplots(figsize=(10, 5))
order_v = ['Known_Tumor', 'Verify_cl5', 'Verify_cl7', 'Macrophage_assigned', 'Known_Immune']
sns.violinplot(data=adata_focus.obs, x='verify_group', y='cnv_score',
               order=order_v, ax=ax, inner='box', cut=0)
ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.set_title('CNV Score: Known Tumor vs Known Immune vs Clusters 5/7')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(FIG_DIR / 'violin_cnv_verify.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistical test
from scipy.stats import mannwhitneyu
for cl in ['5', '7']:
    cl_scores = adata.obs.loc[adata.obs['leiden'] == cl, 'cnv_score']
    tumor_scores = adata.obs.loc[adata.obs['leiden'].isin(['8', '9']), 'cnv_score']
    immune_scores = adata.obs.loc[adata.obs['leiden'].isin(['0', '3', '6']), 'cnv_score']
    _, p_tumor = mannwhitneyu(cl_scores, tumor_scores, alternative='two-sided')
    _, p_immune = mannwhitneyu(cl_scores, immune_scores, alternative='two-sided')
    print(f'Cluster {cl}: median={cl_scores.median():.4f}, '
          f'vs Tumor p={p_tumor:.2e}, vs Immune p={p_immune:.2e}')

### 8. Conclusion

In [ ]:
# Final classification
tumor_median = adata.obs.loc[adata.obs['cell_type'] == 'Tumor', 'cnv_score'].median()
immune_median = adata.obs.loc[adata.obs['cell_type'].isin(['Endothelial', 'NK/T', 'Pericyte']), 'cnv_score'].median()
threshold = (tumor_median + immune_median) / 2

for cl in ['5', '7', '2', '4']:
    med = adata.obs.loc[adata.obs['leiden'] == cl, 'cnv_score'].median()
    verdict = 'TUMOR-like' if med > threshold else 'IMMUNE-like'
    print(f'Cluster {cl} ({adata.obs.loc[adata.obs["leiden"]==cl, "cell_type"].iloc[0]}): '
          f'median CNV={med:.4f} → {verdict}')

print(f'\nReference: Tumor median={tumor_median:.4f}, Immune median={immune_median:.4f}, threshold={threshold:.4f}')

### Further analysis: CNV signals in neuronal clusters 15 and 18

The chromosome heatmap shows clear CNV signals in clusters 15 and 18, particularly on chr7. Check whether these clusters include tumor cells.

In [ ]:
# Summarize clusters 15 and 18.
for cl in ['15', '18']:
    mask = adata.obs['leiden'] == cl
    ct = adata.obs.loc[mask, 'cell_type'].value_counts()
    n = mask.sum()
    med = adata.obs.loc[mask, 'cnv_score'].median()
    print(f"=== Cluster {cl} (n={n}, median CNV={med:.4f}) ===")
    print(ct)
    print()

In [ ]:
# Compare CNV score distributions in neuronal, tumor and immune clusters.
from scipy.stats import mannwhitneyu

suspect_cls = ['15', '18']
known_tumor_cls = ['8', '9', '17']
known_immune_cls = ['0', '3', '6']

tumor_scores = adata.obs.loc[adata.obs['leiden'].isin(known_tumor_cls), 'cnv_score']
immune_scores = adata.obs.loc[adata.obs['leiden'].isin(known_immune_cls), 'cnv_score']

# Statistical test
for cl in suspect_cls:
    cl_scores = adata.obs.loc[adata.obs['leiden'] == cl, 'cnv_score']
    _, p_vs_tumor = mannwhitneyu(cl_scores, tumor_scores, alternative='two-sided')
    _, p_vs_immune = mannwhitneyu(cl_scores, immune_scores, alternative='two-sided')
    print(f"Cluster {cl}: median={cl_scores.median():.4f}, "
          f"vs Tumor p={p_vs_tumor:.2e}, vs Immune p={p_vs_immune:.2e}")

# Violin plot
groups = {
    'Known_Tumor': known_tumor_cls,
    'Neuronal_cl15': ['15'],
    'Neuronal_cl18': ['18'],
    'Known_Immune': known_immune_cls,
}
records = []
for label, cls in groups.items():
    mask = adata.obs['leiden'].isin(cls)
    for s in adata.obs.loc[mask, 'cnv_score']:
        records.append({'group': label, 'cnv_score': s})
df_neuro = pd.DataFrame(records)

fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=df_neuro, x='group', y='cnv_score', ax=ax, inner='box', cut=0)
ax.axhline(y=0.0099, color='red', linestyle='--', alpha=0.6, label='threshold')
ax.set_title('CNV Score: Neuronal clusters 15/18 vs Tumor/Immune')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'violin_neuronal_suspect.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Rank all clusters by their summary CNV scores.
summary = (adata.obs.groupby('leiden')['cnv_score']
           .agg(['median', 'mean', 'count'])
           .sort_values('median', ascending=False))
summary['cell_type'] = adata.obs.groupby('leiden')['cell_type'].agg(lambda x: x.value_counts().index[0])
summary['above_threshold'] = summary['median'] > 0.0099
print(summary.to_string())
print(f"\nThreshold = 0.0099 (midpoint of the tumor and immune medians)")

In [ ]:
# chr_pos maps chromosome names to their starting window indices.
chr_pos = adata.uns['cnv']['chr_pos']
n_windows = adata.obsm['X_cnv'].shape[1]

# Construct the (start, end) interval for each chromosome.
chrs_sorted = sorted(chr_pos.keys(), key=lambda x: int(x.replace('chr', '')))
chr_ranges = {}
for i, c in enumerate(chrs_sorted):
    start = chr_pos[c]
    end = chr_pos[chrs_sorted[i+1]] if i+1 < len(chrs_sorted) else n_windows
    chr_ranges[c] = (start, end)

print(f"chr7: windows [{chr_ranges['chr7'][0]}, {chr_ranges['chr7'][1]})")
print(f"chr10: windows [{chr_ranges['chr10'][0]}, {chr_ranges['chr10'][1]})")

# chr7 gain / chr10 loss per cluster
results = []
for cl in sorted(adata.obs['leiden'].unique(), key=int):
    mask = (adata.obs['leiden'] == cl).values
    X_cnv = adata.obsm['X_cnv'][mask]
    s7, e7 = chr_ranges['chr7']
    s10, e10 = chr_ranges['chr10']
    chr7_mean = X_cnv[:, s7:e7].mean()
    chr10_mean = X_cnv[:, s10:e10].mean()
    ct = adata.obs.loc[adata.obs['leiden'] == cl, 'cell_type'].mode()[0]
    results.append({
        'cluster': cl, 'cell_type': ct, 'n': mask.sum(),
        'chr7_gain': round(chr7_mean, 4),
        'chr10_loss': round(chr10_mean, 4),
    })

df_chr = pd.DataFrame(results).sort_values('chr7_gain', ascending=False)
print("\n--- Chr7 gain / Chr10 loss per cluster ---")
print(df_chr.to_string(index=False))

In [ ]:
# Scatter plot of chr7 gain versus chr10 loss, with one point per cluster.
fig, ax = plt.subplots(figsize=(9, 7))

# Map groups to colors.
color_map = {
    'Tumor': 'red', 'Prol.tumor': 'salmon',
    'Neuronal': 'orange', 'Oligodendrocyte': 'purple',
    'Macrophage': 'steelblue', 'Microglia': 'dodgerblue',
    'Monocyte': 'skyblue', 'NK/T': 'cyan',
    'Endothelial': 'green', 'Pericyte': 'olive', 'Neutrophil': 'gray'
}

for _, row in df_chr.iterrows():
    c = color_map.get(row['cell_type'], 'black')
    ax.scatter(row['chr7_gain'], row['chr10_loss'],
               s=np.sqrt(row['n'])*3, c=c, edgecolors='k', linewidth=0.5, alpha=0.8)
    ax.annotate(f"cl{row['cluster']}", (row['chr7_gain'], row['chr10_loss']),
                fontsize=8, ha='left', va='bottom')

# Reference lines
ax.axvline(x=0.008, color='red', linestyle='--', alpha=0.4, label='chr7 gain cutoff')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('Mean chr7 CNV signal (gain →)')
ax.set_ylabel('Mean chr10 CNV signal (← loss)')
ax.set_title('Chr7 Gain vs Chr10 Loss per Cluster (bubble size = cell count)')

# Legend for cell types
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=v, markersize=8, label=k)
           for k, v in color_map.items() if k in df_chr['cell_type'].values]
ax.legend(handles=handles, loc='upper left', fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(FIG_DIR / 'scatter_chr7_chr10_per_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Inspect the CNV score distribution of cluster 0 (Microglia) for bimodality.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, cl, title in zip(axes, ['0', '15', '16'],
                          ['cl0 Microglia (n=12350)', 'cl15 Neuronal (n=1918)', 'cl16 Oligo (n=2844)']):
    scores = adata.obs.loc[adata.obs['leiden'] == cl, 'cnv_score']
    ax.hist(scores, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
    ax.axvline(x=0.0099, color='red', linestyle='--', alpha=0.6, label='threshold')
    ax.set_title(title)
    ax.set_xlabel('cnv_score')
    above = (scores > 0.0099).sum()
    ax.text(0.95, 0.95, f'{above}/{len(scores)} above thr\n({above/len(scores)*100:.1f}%)',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig(FIG_DIR / 'hist_suspect_cnv_dist.png', dpi=150, bbox_inches='tight')
plt.show()

### CNV validation summary and proposed annotation changes

**Annotations supported by CNV validation**
- ✅ Macrophage (cl2/4/5/7): normal diploid profiles support the annotation.
- ✅ Tumor (cl8/9/17): strong CNV signals support the annotation.
- ✅ Prol.tumor (cl1): moderate CNV signals support the annotation.
- ✅ Other immune/stromal cells (cl3/6/10/11/12/13/14): normal diploid profiles.

**Proposed annotation changes**
- ⚠️ **cl15 Neuronal (n=1918)**: 96.1% above threshold with chr7 gain. **Reassign to NPC-like Tumor.**
- ⚠️ **cl18 Neuronal (n=314)**: the same pattern as cl15. **Reassign to NPC-like Tumor.**
- ⚠️ **cl16 Oligodendrocyte (n=2844)**: 99.3% above threshold with chr7 gain. **Reassign to OPC-like Tumor.**
- ⚠️ **cl0 Microglia (n=12350)**: 34.3% above threshold with a bimodal distribution. **Subcluster to separate microglia from admixed tumor cells.**

**Biological interpretation:** GBM tumor cells can express four transcriptional programs (Neftel 2019): MES-like, AC-like, NPC-like (neuronal-like) and OPC-like (oligodendrocyte-like). This analysis interprets cl15/18 as tumor cells with the NPC-like program and cl16 as tumor cells with the OPC-like program.

In [ ]:
# Prepare revised labels in a separate column for downstream use; this cell leaves adata2 unchanged on disk.
cell_type_corrected = adata.obs['cell_type'].astype(str).copy()

# cl15/18 → NPC-like Tumor
cell_type_corrected.loc[(adata.obs['leiden'] == '15')] = 'NPC-like Tumor'
cell_type_corrected.loc[(adata.obs['leiden'] == '18')] = 'NPC-like Tumor'

# cl16 → OPC-like Tumor
cell_type_corrected.loc[(adata.obs['leiden'] == '16')] = 'OPC-like Tumor'

# Split cl0 using the cnv_score threshold.
cl0_mask = adata.obs['leiden'] == '0'
cl0_tumor_mask = cl0_mask & (adata.obs['cnv_score'] > 0.0099)
cell_type_corrected.loc[cl0_tumor_mask] = 'Tumor (from cl0)'

adata.obs['cell_type_cnv_corrected'] = pd.Categorical(cell_type_corrected)

# Summarize the assignments.
print("=== Annotation comparison ===")
print("\nOriginal annotations:")
print(adata.obs['cell_type'].value_counts().to_string())
print("\nCNV-corrected annotations:")
print(adata.obs['cell_type_cnv_corrected'].value_counts().to_string())

# Count the cells whose labels changed.
changed = (adata.obs['cell_type'].astype(str) != adata.obs['cell_type_cnv_corrected'].astype(str)).sum()
print(f"\nCells relabeled: {changed} ({changed/len(adata)*100:.1f}%)")

In [ ]:
# UMAP: original and CNV-corrected annotations
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sc.pl.umap(adata, color='cell_type', ax=axes[0], show=False, title='Original annotations', legend_loc='on data', legend_fontsize=7)
sc.pl.umap(adata, color='cell_type_cnv_corrected', ax=axes[1], show=False, title='CNV-corrected annotations', legend_loc='on data', legend_fontsize=7)

plt.tight_layout()
plt.savefig(FIG_DIR / 'umap_annotation_correction.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Replace cell_type and save.
adata.obs['cell_type_original'] = adata.obs['cell_type'].copy()  # Preserve the original labels.
adata.obs['cell_type'] = adata.obs['cell_type_cnv_corrected'].copy()

# Save adata2.h5ad.
adata.write_h5ad('adata2.h5ad')
print("Saved adata2.h5ad")
print("\nCorrected cell_type:")
print(adata.obs['cell_type'].value_counts().to_string())

## Overview plots after annotation correction

In [ ]:
# Overview UMAP for the corrected cell_type
fig, axes = plt.subplots(1, 2, figsize=(24, 10))

# Left: corrected labels
sc.pl.umap(adata, color='cell_type', ax=axes[0], show=False,
           title='Cell type (CNV-corrected)', frameon=False, legend_loc='right margin')
# Right: Leiden clusters for comparison
sc.pl.umap(adata, color='leiden', ax=axes[1], show=False,
           title='Leiden clusters', frameon=False, legend_loc='on data',
           legend_fontsize=8)

plt.tight_layout()
fig.savefig(f'{FIG_DIR}/umap_corrected_celltype.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: umap_corrected_celltype.png")

In [ ]:
# Define marker genes as in notebook 2.
gbm_markers = {
    "Tumor":          ["GFAP", "PTPRZ1", "SCG3", "C1orf61", "SOX2", "SOX6", "BCAN"],
    "Prol.tumor":     ["MKI67", "TOP2A", "CENPF", "PBK"],
    "NPC-like":       ["SYT1", "STMN2", "RBFOX3", "SLC17A7", "GAD1"],
    "OPC-like":       ["MAG", "CLDN11", "MOG", "APLP1"],
    "NK/T":           ["CD3D", "CD3E", "TRBC2", "CD2"],
    "Pericyte":       ["COL1A2", "PDGFRB", "COL3A1"],
    "Endothelial":    ["ESAM", "FLT1", "VWF", "EGFL7"],
    "Monocyte":       ["FCN1", "VCAN", "S100A8", "S100A9"],
    "Microglia":      ["TMEM119", "CX3CR1", "P2RY12", "TREM2"],
    "Macrophage":     ["CD68", "LYZ", "C1QA", "C1QB", "APOE", "SPP1"],
    "Neutrophil":     ["FCGR3B", "CSF3R", "CXCR2"],
}

# Filter out missing genes.
markers_filt = {}
for ct, genes in gbm_markers.items():
    found = [g for g in genes if g in adata.var_names]
    markers_filt[ct] = found
    miss = set(genes) - set(found)
    if miss:
        print(f'⚠️ {ct}: missing {miss}')

print(f'Available markers: {sum(len(v) for v in markers_filt.values())}')

In [ ]:
# Recompute the dendrogram after the number of cell_type categories changes from 11 to 12.
sc.tl.dendrogram(adata, groupby='cell_type')

# Dot plot of marker genes by corrected cell type.
sc.pl.dotplot(adata, markers_filt, groupby='cell_type',
              standard_scale='var', dendrogram=True,
              save='_corrected_celltype_markers.png')
print("✅ Saved: dotplot_corrected_celltype_markers.png")

In [ ]:
# Compare cell proportions before and after annotation correction.
import pandas as pd

df_orig = adata.obs['cell_type_original'].value_counts(normalize=True).rename('original')
df_corr = adata.obs['cell_type'].value_counts(normalize=True).rename('corrected')
df_comp = pd.concat([df_orig, df_corr], axis=1).fillna(0).sort_values('corrected', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: original labels
df_orig_sorted = df_comp['original'].sort_values()
df_orig_sorted.plot.barh(ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Original annotation')
axes[0].set_xlabel('Fraction')

# Right: corrected labels
df_corr_sorted = df_comp['corrected'].sort_values()
df_corr_sorted.plot.barh(ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('CNV-corrected annotation')
axes[1].set_xlabel('Fraction')

plt.tight_layout()
fig.savefig(f'{FIG_DIR}/barplot_celltype_proportion_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: barplot_celltype_proportion_comparison.png")

In [ ]:
### Merge Tumor (from cl0) into Tumor

In [ ]:
# Merge the "Tumor (from cl0)" label into "Tumor".
ct = adata.obs['cell_type'].astype(str)
ct[ct == 'Tumor (from cl0)'] = 'Tumor'
adata.obs['cell_type'] = pd.Categorical(ct)
print(adata.obs['cell_type'].value_counts())

In [ ]:
# Save the merged annotations.
adata.write_h5ad('adata2.h5ad')
print(f"✅ Saved adata2.h5ad, {adata.n_obs} cells × {adata.n_vars} genes, {adata.obs['cell_type'].nunique()} cell types")

In [ ]:
# UMAP after merging the tumor labels
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sc.pl.umap(adata, color='cell_type', ax=axes[0], show=False, title='Cell type (11 types)', frameon=False)
sc.pl.umap(adata, color='leiden', ax=axes[1], show=False, title='Leiden clusters', frameon=False, legend_loc='on data')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/umap_final_11types.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: umap_final_11types.png")